In [2]:
import pandas as pd
import numpy as np

In [4]:
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

In [6]:
deliveries.head()


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


In [5]:
matches.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


In [7]:
matches.isnull().sum()

,0
id,0
season,0
city,51
date,0
match_type,0
player_of_match,5
venue,0
team1,0
team2,0
toss_winner,0


In [8]:
final_df = deliveries.merge(
    matches[['id', 'winner']],
    left_on='match_id',
    right_on='id'
)

In [9]:
final_df = final_df[
    final_df['inning'] == 2
]

In [10]:
final_df['current_score'] = (
    final_df.groupby('match_id')['total_runs']
    .cumsum()
)

In [11]:
final_df['is_wicket'] = (
    final_df['player_dismissed']
    .notna()
    .astype(int)
)

final_df['wickets'] = (
    final_df.groupby('match_id')['is_wicket']
    .cumsum()
)

In [12]:
final_df['balls_bowled'] = (
    (final_df['over'] - 1) * 6
    + final_df['ball']
)

final_df['balls_left'] = (
    120 - final_df['balls_bowled']
)

In [14]:
total_score_df = final_df.groupby('match_id')['total_runs'].sum().reset_index()

total_score_df['target'] = (
    total_score_df['total_runs'] + 1
)

In [15]:
final_df = final_df.merge(
    total_score_df[['match_id', 'target']],
    on='match_id'
)

In [16]:
final_df['runs_left'] = (
    final_df['target']
    - final_df['current_score']
)

In [17]:
final_df['rrr'] = (
    final_df['runs_left'] * 6
    / final_df['balls_left']
)

In [18]:
final_df['balls_bowled'] = (
    (final_df['over'] - 1) * 6
    + final_df['ball']
)

In [19]:
final_df['current_score'] = (
    final_df.groupby('match_id')['total_runs']
    .cumsum()
)

In [20]:
final_df['crr'] = (
    final_df['current_score'] * 6
    / final_df['balls_bowled']
)

In [21]:
final_df['result'] = (
    final_df['batting_team']
    == final_df['winner']
).astype(int)

In [23]:
final_df['is_wicket'] = (
    final_df['player_dismissed']
    .notna()
    .astype(int)
)

In [24]:
final_df['wickets'] = (
    final_df.groupby('match_id')['is_wicket']
    .cumsum()
)

In [25]:
final_df['wickets_left'] = (
    10 - final_df['wickets']
)

In [27]:
final_df = final_df.replace([np.inf, -np.inf], np.nan)

final_df = final_df.dropna()

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = final_df[
    [
        'runs_left',
        'balls_left',
        'wickets_left',
        'crr',
        'rrr'
    ]
]

y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [29]:
model.score(X_test, y_test)

0.4

In [31]:
sample = pd.DataFrame(
    [[30,18,7,9,10]],
    columns=[
        'runs_left',
        'balls_left',
        'wickets_left',
        'crr',
        'rrr'
    ]
)

model.predict_proba(sample)

array([[0.39, 0.61]])

In [34]:
sample = pd.DataFrame(
    [[52,24,3,10,13]],
    columns=[
        'runs_left',
        'balls_left',
        'wickets_left',
        'crr',
        'rrr'
    ]
)

model.predict_proba(sample)

array([[0.41, 0.59]])

In [35]:
model.score(X_test, y_test)

0.4

In [36]:
import pickle

pickle.dump(
    model,
    open('ipl_model.pkl', 'wb')
)